# Descriptive Statistics and CRM Data Overview

This notebook provides a structured overview of the cleaned CRM datasets before deeper analysis.

### Main tasks
- summarize call activity and call outcomes
- examine manager workload and contact growth
- review marketing spend, impressions, and clicks
- inspect deal values, SLA, stages, lead quality, products, and payment types
- identify skewed distributions, outliers, and data-quality limitations

> **Data note:** the original CRM datasets are not included in the public repository.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "helpers.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from helpers import (
    colors,
    descriptive_stats,
    cat_stats,
    date_stats,
    plot_distributions,
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"


## Load Processed Data

In [ ]:
calls = pd.read_pickle(PROCESSED_DIR / 'calls_clean.pkl')
deals = pd.read_pickle(PROCESSED_DIR / 'deals_clean.pkl')
spend = pd.read_pickle(PROCESSED_DIR / 'spend_clean.pkl')
contacts = pd.read_pickle(PROCESSED_DIR / 'contacts_clean.pkl')

## 1. Calls

In [ ]:
print('Data overview:')
print(calls.info())

In [ ]:
calls.head()

In [ ]:
descriptive_stats(calls, exclude=['id', 'contactid'])

In [ ]:
cat_stats(calls, table_name='Calls', exclude=['id', 'contactid'])

In [ ]:
plot_distributions(calls, cols=['call_type', 'call_status', 'call_duration'], ncols=2)

In [ ]:
# Overall share of successful calls
calls['is_successful'].value_counts(normalize=True) * 100

In [ ]:
# Distribution of successful-call duration
successful_calls = calls[calls['is_successful'] == True]

plt.figure(figsize=(8, 4))
sns.histplot(successful_calls['call_duration'], bins=40, color=colors['accent'])
plt.title('Successful Call Duration Distribution')
plt.xlabel('Duration (seconds)')
plt.show()

print(successful_calls['call_duration'].describe().round(2))

### Calls — Key Findings

**Volume**
- The dataset contains **92,599 calls** handled by 33 managers.
- The top managers account for a substantial share of call activity, while a long tail of managers handles less than 1% each.

**Call type**
- **90.5%** of calls are Outbound, indicating a strongly proactive sales process.
- Inbound calls represent only **3.3%**.

**Call outcome**
- 75% of calls are `Attended Dialled`.
- 15.3% are `Unattended Dialled`.
- 6.2% are `Missed`.
- Roughly **21%** of calls do not result in customer contact.

**Duration**
- Median duration is **9 seconds**, while the mean is much higher because of a long right tail.
- Successful calls have a median duration of **227 seconds (~4 minutes)**.
- 75% of successful calls last up to **574 seconds (~10 minutes)**.

The large difference between median duration for all calls and successful calls suggests that longer conversations are characteristic of successful customer contact.

## 2. Contacts

In [ ]:
print('Data overview:')
print(contacts.info())

In [ ]:
contacts.head()

In [ ]:
descriptive_stats(contacts, exclude=['id'])

In [ ]:
cat_stats(contacts, table_name='Contacts', exclude=['id'])

In [ ]:
vc = contacts['contact_owner_name'].value_counts(normalize=True) * 100

plt.figure(figsize=(7, 5))
ax = sns.barplot(x=vc.values, y=vc.index, color=colors['accent'])

ax.bar_label(ax.containers[0], fmt='%.1f%%', padding=3)

plt.title('Contact Distribution by Manager')
plt.xlabel('%')
plt.ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
date_stats(contacts, ['created_time', 'modified_time'], table_name='Contacts')

### Contacts — Key Findings

**Manager workload**
- 27 managers handle **18,510 contacts**.
- Workload is uneven: the top five managers account for **45.9%** of all contacts.
- Several managers form a long tail with less than 0.2% of contacts each.

**Contact creation**
- The dataset covers almost one year, from **27 Jun 2023 to 21 Jun 2024**.
- New contacts increased from about 630 in Jul 2023 to a peak of **2,531 in Apr 2024**, indicating strong business growth.
- Jun 2024 is incomplete because the dataset ends on 21 Jun.

**Record modification**
- Apr 2024 is also the peak month for CRM modifications.
- Modification activity remains high in Jun 2024 despite the partial month, suggesting continued work on previously created contacts.

## 3. Marketing Spend

In [ ]:
print('Data overview:')
print(spend.info())

In [ ]:
spend.head()

In [ ]:
print(f"Total marketing spend: €{spend['spend'].sum():,.0f}")
print(f"Total impressions: {spend['impressions'].sum():,.0f}")
print(f"Total clicks: {spend['clicks'].sum():,.0f}")

In [ ]:
descriptive_stats(spend, exclude=['campaign', 'ad_group', 'ad', 'is_active', 'is_test'])

In [ ]:
cat_stats(spend, table_name='Spend', exclude=['date', 'ad', 'ad_group'])

In [ ]:
# Date range
date_stats(spend, ['date'], table_name='Spend', plot=False)

### Marketing Spend — Key Findings

- The dataset contains **19,862 records** covering **03 Jul 2023–21 Jun 2024**.
- Total tracked marketing spend is **€149,523**.
- Campaigns generated approximately **51 million impressions** and **498,455 clicks**.
- The data contains 14 marketing sources; the largest include Google Ads, Facebook Ads, YouTube Ads, TikTok Ads, and Telegram posts.

## 4. Deals

In [ ]:
print('Data overview:')
print(deals.info())

In [ ]:
deals.head()

In [ ]:
descriptive_stats(deals, exclude=['id', 'contact_name'])

In [ ]:
# Inspect the upper tail of SLA response time
sla_99 = deals['sla_minutes'].quantile(0.99)
outliers = deals[deals['sla_minutes'] > sla_99]
print(f'Rows above the 99th percentile: {len(outliers)}')
print(f'Share of non-missing SLA values: {len(outliers) / deals["sla_minutes"].notna().sum() * 100:.1f}%')
print(f'99th percentile threshold: {sla_99:.0f} min')

In [ ]:
plt.figure(figsize=(10, 4))
sns.boxplot(x=deals['sla_minutes'].dropna(), color=colors['accent'])
plt.title('SLA Response Time (minutes)', fontweight='bold')
plt.xlabel('Minutes')
plt.show()

In [ ]:
sla_clean = deals[deals['sla_minutes'] <= sla_99]['sla_minutes']
print('SLA up to the 99th percentile:')
print(sla_clean.describe().round(1))

In [ ]:
# Numerical fields
plot_distributions(deals, cols=['initial_amount_paid', 'offer_total_amount', 'deal_duration_days'], ncols=2)

In [ ]:
# Categorical fields
cat_stats(deals, table_name='Deals', exclude=['id', 'contact_name', 'city', 'content', 'page', 'term'])

In [ ]:
# Categorical fields
plot_distributions(deals, cols=['stage', 'lost_reason', 'quality', 'product', 'education_type', 'payment_type', 'course_duration', 'months_of_study', 'source', 'level_of_deutsch'], ncols=2)

### Deals — Key Findings

**Numerical fields**
- **SLA:** median response time is about **330 minutes (5.5 hours)**. The mean is substantially higher because of extreme outliers.
- After limiting SLA to the 99th percentile, the median remains about **5.4 hours**; 25% of leads receive a response within about 72 minutes, while 75% wait up to about 15 hours.
- **Initial Amount Paid:** median €1,000.
- **Offer Total Amount:** median €11,000.
- **Deal Duration:** median 4 days, with a long right tail.

**Categorical fields**
- **Lead Quality:** 62% of deals are classified as `E - Non Qualified` or `D - Non Target`; only 2.1% are `A - High`.
- **Stage:** 70.6% of deals are Lost; `Payment Done` represents 856 records (4.3%).
- **Source:** the largest sources are Facebook Ads, Google Ads, and TikTok Ads.
- **Product:** Digital Marketing accounts for the largest share of deals, followed by UX/UI Design.
- **Education Type:** among populated records, Morning is the dominant format.
- **Payment Type:** Recurring Payments dominate over One Payment.
- **Data limitation:** `Level of Deutsch` and `City` contain a very high share of `Unknown` values, so conclusions based on these fields should be treated cautiously.